In [1]:
import numpy as np
import torch 
import torch.nn as nn
import time 

In [2]:
np.random.seed(42)
torch.manual_seed(42)

In [3]:
def quantize_matrix(x: np.ndarray) -> tuple[np.ndarray,float,int]:
    """
    Quantizes a float32 array into INT8 tensors 
    
    """
    
    q_min,q_max = -128,127
    x_min , x_max = float(x.min()) , float(x.max())
    
    if x_min == x_max:
        return np.zeros_like(x,dtype=np.int8), 1.0, 0
    
    scale = (x_max - x_min) / (q_max - q_min)
    
    zero_point = np.round(-x_min/ scale) + q_min
    zero_point = int(np.clip(zero_point,q_min,q_max))
    
    q_scaled = np.round(x/scale) + zero_point
    quantized = np.clip(q_scaled , q_min,q_max).astype(np.int8)
    
    return quantized ,scale , zero_point

In [4]:
def dequantize_matrix(q: np.ndarray ,scale:float , zero_point: int)-> np.ndarray:
    """
    Reconstructs an int8 matrix into its original float32 continuos sprctrum
    """
    return scale*(q.astype(np.float32) - zero_point)

In [5]:
# Generate a random 4x4 matrix representing weights with a float32 layout
original_matrix = np.random.randn(4, 4).astype(np.float32) * 5

# Process through our scratch quantization engine
quantized_matrix, scale, zero_point = quantize_matrix(original_matrix)

# Reconstruct back to float32 space
reconstructed_matrix = dequantize_matrix(quantized_matrix, scale, zero_point)

# Display the numerical mapping profiles
print("=== QUANTIZATION CALIBRATION METADATA ===")
print(f"Scale Factor (S): {scale:.6f}")
print(f"Zero Point (Z):   {zero_point}")
print("=" * 41 + "\n")

print("--- 1. ORIGINAL MATRIX (FP32) ---")
print(original_matrix)
print("\n--- 2. QUANTIZED MATRIX (INT8) ---")
print(quantized_matrix)
print("\n--- 3. RECONSTRUCTED MATRIX (FP32) ---")
print(reconstructed_matrix)

=== QUANTIZATION CALIBRATION METADATA ===
Scale Factor (S): 0.068480
Zero Point (Z):   12

--- 1. ORIGINAL MATRIX (FP32) ---
[[ 2.4835708 -0.6913215  3.238443   7.615149 ]
 [-1.1707668 -1.1706848  7.896064   3.8371735]
 [-2.3473718  2.7128003 -2.3170884 -2.3286488]
 [ 1.2098113 -9.5664015 -8.62459   -2.8114376]]

--- 2. QUANTIZED MATRIX (INT8) ---
[[  48    2   59  123]
 [  -5   -5  127   68]
 [ -22   52  -22  -22]
 [  30 -128 -114  -29]]

--- 3. RECONSTRUCTED MATRIX (FP32) ---
[[ 2.465289   -0.68480253  3.218572    7.601308  ]
 [-1.1641643  -1.1641643   7.875229    3.8348942 ]
 [-2.3283286   2.7392101  -2.3283286  -2.3283286 ]
 [ 1.2326446  -9.587235   -8.628511   -2.8076904 ]]


In [6]:
#ERROR Computation 

mse = np.mean((original_matrix - reconstructed_matrix)**2)

In [7]:
signal_power = np.mean(original_matrix**2)

In [12]:
sqnr_db = 10 * np.log10(signal_power/ (mse + 1e-10))

In [13]:
print("=== QUANTIZATION FIDELITY METRICS ===")
print(f"Mean Squared Error (MSE):                   {mse:.6f}")
print(f"Signal-to-Quantization-Noise Ratio (SQNR):  {sqnr_db:.2f} dB")
print("========================================")

=== QUANTIZATION FIDELITY METRICS ===
Mean Squared Error (MSE):                   0.000229
Signal-to-Quantization-Noise Ratio (SQNR):  49.85 dB


In [14]:
fp32_memory = original_matrix.nbytes
int8_memory = quantized_matrix.nbytes

In [15]:
compression_ratio = fp32_memory / int8_memory
memory_saved_pct = (1 - (int8_memory / fp32_memory)) * 100

In [16]:
print("=== MEMORY FOOTPRINT ANALYSIS ===")
print(f"Original FP32 Matrix Size:  {fp32_memory} bytes")
print(f"Quantized INT8 Matrix Size: {int8_memory} bytes")
print("-" * 33)
print(f"Compression Ratio:          {compression_ratio}x")
print(f"Total Memory Saved:         {memory_saved_pct:.1f}%")
print("=================================")

=== MEMORY FOOTPRINT ANALYSIS ===
Original FP32 Matrix Size:  64 bytes
Quantized INT8 Matrix Size: 16 bytes
---------------------------------
Compression Ratio:          4.0x
Total Memory Saved:         75.0%


# Pytorch implementation 

In [17]:
import io 
import time 

In [20]:
class LargeFeedforward(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(4096,4096)
        
    def forward(self,x):
        return self.fc(x)

In [21]:
model_fp32 = LargeFeedforward()

In [22]:
!pip install torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 3.7 MB/s  0:00:00 eta 0:00:01


In [24]:
from torchao.quantization import quantize_ , Int8DynamicActivationInt8WeightConfig
import copy

In [25]:
model_int8 = copy.deepcopy(model_fp32)

In [26]:
quantize_(model_int8, Int8DynamicActivationInt8WeightConfig())

/opt/miniconda3/envs/ml_month3/lib/python3.10/site-packages/torchao/dtypes/utils.py:89: UserWarning: Deprecation: PlainLayout is deprecated and will be removed in a future release of torchao, see https://github.com/pytorch/ao/issues/2752 for more details
  warnings.warn(
/opt/miniconda3/envs/ml_month3/lib/python3.10/site-packages/torchao/dtypes/uintx/plain_layout.py:82: UserWarning: Deprecation: PlainAQTTensorImpl is deprecated and will be removed in a future release of torchao, see https://github.com/pytorch/ao/issues/2752 for more details
  warnings.warn(
/opt/miniconda3/envs/ml_month3/lib/python3.10/site-packages/torchao/dtypes/affine_quantized_tensor.py:116: UserWarning: Deprecation: AffineQuantizedTensor is deprecated and will be removed in a future release of torchao, see https://github.com/pytorch/ao/issues/2752 for more details
  warnings.warn(


In [27]:
def get_model_size_mb(model: nn.Module) -> float:
    buffer = io.BytesIO()
    torch.save(model.state_dict(), buffer)
    return buffer.tell() / (1024 ** 2)

fp32_size = get_model_size_mb(model_fp32)
int8_size = get_model_size_mb(model_int8)

In [28]:
print("\n=== MODEL SIZE COMPARISON ===")
print(f"FP32 Model Size: {fp32_size:.2f} MB")
print(f"INT8 Model Size: {int8_size:.2f} MB")
print(f"Compression:     {fp32_size / int8_size:.2f}x")
print("=============================\n")

# 4. Measure Inference Speed (CPU)
input_tensor = torch.randn(64, 4096) # Simulating a batch of 64 tokens

print("Running Inference Benchmarks (100 iterations)...")

# Warm-up
for _ in range(5):
    _ = model_fp32(input_tensor)
    _ = model_int8(input_tensor)

# Benchmark FP32
start_time = time.perf_counter()
for _ in range(100):
    _ = model_fp32(input_tensor)
fp32_time = time.perf_counter() - start_time

# Benchmark INT8
start_time = time.perf_counter()
for _ in range(100):
    _ = model_int8(input_tensor)
int8_time = time.perf_counter() - start_time

print("=== INFERENCE LATENCY ===")
print(f"FP32 Time (100 runs): {fp32_time:.3f} seconds")
print(f"INT8 Time (100 runs): {int8_time:.3f} seconds")
if int8_time < fp32_time:
    print(f"Speedup:              {fp32_time / int8_time:.2f}x Faster")
else:
    print("Speedup:              No speedup (Hardware specific)")
print("=========================")


=== MODEL SIZE COMPARISON ===
FP32 Model Size: 64.02 MB
INT8 Model Size: 16.03 MB
Compression:     3.99x

Running Inference Benchmarks (100 iterations)...
=== INFERENCE LATENCY ===
FP32 Time (100 runs): 2.778 seconds
INT8 Time (100 runs): 23.819 seconds
Speedup:              No speedup (Hardware specific)
